# Clase 6 — Laboratorio de Retrieval y Evaluación

Esta clase toma el índice de la Clase 5 y lo convierte en un componente **observable, evaluable y mantenible**. El objetivo no es obtener una demo que “parece responder bien”, sino medir si recupera evidencia correcta y detectar por qué falla.

## Resultado de la clase

Al finalizar vas a tener:

- loaders normalizados para Markdown, texto, JSON y PDF;
- chunking e ingesta incremental;
- caché de embeddings con métricas de hits y misses;
- indexación en batches;
- filtros y payload indexes;
- actualización y eliminación segura de documentos;
- golden dataset;
- Precision@k, Recall@k, Hit Rate, MRR y latencia;
- un umbral para casos sin evidencia suficiente;
- un reporte consolidado listo para la Clase 7 de RAG.

---

### Contrato de continuidad

- Las notebooks de las clases 1 y 2 son la **base publicada e inmutable**.
- Esta clase continúa sobre `ai_agent_project/`; no crea un proyecto paralelo.
- Incremento: **ingesta, filtros y evaluación de retrieval**.
- El modo predeterminado es local/simulado y no consume APIs por accidente.
- Los modos reales requieren variables de entorno explícitas.

La notebook separa experimentos visibles de módulos reutilizables y deja un reporte consolidado para la clase siguiente.


## Mapa del laboratorio

1. Recuperar el contrato de la Clase 5.
2. Cargar documentos heterogéneos.
3. Normalizar metadata y trazabilidad.
4. Aplicar caché y batch processing.
5. Probar filtros, actualización y eliminación.
6. Evaluar con un golden dataset.
7. Analizar casos sin resultados suficientes.
8. Persistir el reporte y preparar el handoff a RAG.

La notebook puede ejecutarse con Qdrant o con el backend compatible en memoria. El reporte registra cuál se utilizó.

### Patrón de trabajo de esta notebook

En cada incremento vamos a seguir la misma secuencia que quedó validada en la Clase 3:

1. **comprender el problema y el contrato**;
2. **construir una versión visible y mínima**;
3. **probar un caso exitoso y un borde relevante**;
4. **persistir la implementación en `src/`**;
5. **importar el módulo generado y volver a verificarlo**.

Los servicios externos permanecen desactivados por defecto. Las celdas que usan APIs, Ollama, Qdrant Server u otros servicios requieren una habilitación explícita.

### Caso transversal — agente de gestión asistida de emails

La Clase 5 preparó las políticas internas. Ahora construimos el componente que permitirá responder una pregunta clave:

> **Dado un email recibido, ¿qué información interna necesito recuperar para responderlo correctamente?**

Todavía no actuamos sobre Gmail. En esta clase nos concentramos en **retrieval y evaluación**: recuperar chunks relevantes, aplicar filtros y medir si el documento correcto aparece entre los primeros resultados.

**Rol de esta clase en el caso:** `email/consulta → retrieval → evidencia relevante`.

In [1]:
# Prepara las rutas y utilidades compartidas del proyecto incremental.
# Esta celda no instala dependencias: crea carpetas, configura imports locales
# y define helpers que usaremos durante toda la notebook.
from pathlib import Path
import hashlib
import importlib
import importlib.util
import json
import os
import re
import statistics
import subprocess
import sys
import textwrap
import time
from dotenv import load_dotenv
load_dotenv("ai_agent_project/.env")

PROJECT_DIR = Path("ai_agent_project")
SRC_DIR = PROJECT_DIR / "src" / "ai_agent_course"
DATA_DIR = PROJECT_DIR / "data"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
TESTS_DIR = PROJECT_DIR / "tests"
CORPUS_DIR = DATA_DIR / "retrieval_corpus"

for directory in (SRC_DIR, DATA_DIR, ARTIFACTS_DIR, TESTS_DIR, CORPUS_DIR):
    directory.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "__init__.py").touch()

if str(PROJECT_DIR / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / "src"))

# Guarda contenido en disco y crea las carpetas necesarias.
def write_file(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).strip() + "\n", encoding="utf-8")
    print("✓", path.relative_to(PROJECT_DIR))

# Muestra una lista de filas como tabla para facilitar la lectura.
# Solo hacemos fallback si falta pandas; otros errores deben quedar visibles.
def show_table(rows, columns=None):
    try:
        import pandas as pd
        frame = pd.DataFrame(rows)
        if columns:
            frame = frame[columns]
        display(frame)
    except ImportError:
        for row in rows:
            print(row)

print(f"Proyecto incremental: {PROJECT_DIR}")

Proyecto incremental: ai_agent_project


### Checkpoint de continuidad

Se verifican los componentes previos antes de sumar el incremento. Un faltante se informa; no se oculta recreando silenciosamente todo el proyecto.

In [2]:
# Comprueba que la Clase 5 dejó los módulos y el reporte necesarios.
# Esta primera verificación informa faltantes antes de entrar al laboratorio.
_continuity_root = PROJECT_DIR
_continuity = ['src/ai_agent_course/embeddings.py', 'src/ai_agent_course/chunking.py', 'src/ai_agent_course/retrieval.py', 'artifacts/class05_report.json']
continuity_status = [{"path": p, "available": (_continuity_root / p).exists()} for p in _continuity]

for item in continuity_status:
    print(("✓" if item["available"] else "·"), item["path"])

missing_prerequisites = [
    item["path"]
    for item in continuity_status
    if not item["available"]
]

if missing_prerequisites:
    print("Aviso: faltan componentes previos:", missing_prerequisites)
else:
    print("Continuidad verificada.")

✓ src/ai_agent_course/embeddings.py
✓ src/ai_agent_course/chunking.py
✓ src/ai_agent_course/retrieval.py
✓ artifacts/class05_report.json
Continuidad verificada.


## 1. Dependencias y continuidad

La ruta normal es ejecutar primero la Clase 5. Esta notebook verifica que el proyecto acumulado conserve los módulos necesarios antes de continuar.

No se reconstruyen silenciosamente `embeddings.py`, `chunking.py` ni `retrieval.py`: si falta alguno, la práctica se detiene para que el problema sea visible.

Qdrant sigue siendo el backend principal del laboratorio; el backend **in-memory** se conserva únicamente como alternativa cuando Qdrant no está disponible.

### ¿Para qué sirven las dependencias opcionales?

- **`reportlab`**: crea un PDF de prueba desde Python.
- **`pypdf`**: lee PDFs y extrae su texto durante la ingesta.
- **`qdrant-client`**: habilita Qdrant como vector store.
- **`sentence-transformers`**: permite usar embeddings locales reales.
- **`pytest`**: ejecuta los tests automatizados.

> **ReportLab y PyPDF cumplen roles distintos:** ReportLab genera el PDF; PyPDF lo lee.  
> No hace falta subir documentos para esta demostración: la notebook crea un corpus pequeño automáticamente.


In [3]:
# Define dependencias opcionales sin instalar paquetes por defecto.
# RUN_INSTALLS=1 permite preparar un entorno vacío de forma explícita.
RUN_INSTALLS = os.getenv("RUN_INSTALLS", "0") == "1"

PACKAGES = [
    "numpy>=1.26",
    "qdrant-client>=1.12",
    "sentence-transformers>=3.0",
    "pypdf>=4.0",
    "reportlab>=4.0",
    "pytest>=8.0",
]

if RUN_INSTALLS:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        *PACKAGES,
    ])
# Detectamos qué capacidades opcionales quedaron disponibles.
# Estos flags permiten degradar funcionalidad de forma explícita sin romper toda la notebook.
HAS_REPORTLAB = importlib.util.find_spec("reportlab") is not None
HAS_PYPDF = importlib.util.find_spec("pypdf") is not None
HAS_QDRANT = importlib.util.find_spec("qdrant_client") is not None

print({
    "reportlab": HAS_REPORTLAB,
    "pypdf": HAS_PYPDF,
    "qdrant_client": HAS_QDRANT,
})


{'reportlab': True, 'pypdf': True, 'qdrant_client': True}


### Precondición acumulativa

Esta notebook ya no recrea silenciosamente los módulos de la Clase 5. El objetivo es trabajar sobre el proyecto acumulado y detectar de inmediato si falta un checkpoint.

In [4]:
# La Clase 6 depende directamente de los componentes construidos en Clase 5.
# Si falta alguno, detenemos la notebook para evitar evaluar un proyecto incompleto.
required_class05_files = [
    SRC_DIR / "embeddings.py",
    SRC_DIR / "chunking.py",
    SRC_DIR / "retrieval.py",
    ARTIFACTS_DIR / "class05_report.json",
]

missing_class05 = [
    str(path.relative_to(PROJECT_DIR))
    for path in required_class05_files
    if not path.exists()
]

if missing_class05:
    raise RuntimeError(
        "Falta ejecutar la Clase 5 antes de continuar. Archivos ausentes: "
        + ", ".join(missing_class05)
    )

print("✓ Precondición Clase 5 verificada")

✓ Precondición Clase 5 verificada


## 2. Crear y cargar un corpus heterogéneo

Vamos a simular una situación habitual: la información útil puede venir en formatos distintos, pero el retriever no debería tener que conocer esas diferencias.

La notebook crea automáticamente ejemplos en Markdown, TXT, JSON y PDF. Después, cada loader transforma su formato de origen al mismo contrato:

`document_id · source · page · text · metadata`

La idea clave es:

**distintos formatos → distintos loaders → un único contrato documental**

Esa normalización evita que chunking, embeddings e indexación tengan una rama distinta para cada tipo de archivo.

> No hace falta subir documentos en esta parte. Los archivos se generan dentro de `CORPUS_DIR` para que todos ejecuten el mismo laboratorio.


In [5]:
# Crea archivos sintéticos de distintos formatos para probar una ingesta heterogénea.
# No hace falta subir documentos: todos terminarán normalizados al mismo contrato.
# 1) Markdown: texto no estructurado con algo de formato.
write_file(
    CORPUS_DIR / "faq_rrhh.md",
    '''
    # Preguntas frecuentes de RRHH

    Para pedir vacaciones, registrá la solicitud con 15 días corridos de anticipación.
    La carga no equivale a aprobación: el líder revisa cobertura y confirma la decisión.

    Para modificar una solicitud aprobada, cancelá la fecha original y generá una nueva solicitud.
    '''
)

# 2) TXT: texto plano.
write_file(
    CORPUS_DIR / "respuesta_incidentes.txt",
    '''
    Si se expone una credencial, cambiá la contraseña, revocá sesiones activas y abrí
    inmediatamente un incidente de seguridad. Informá sistema, horario y canal de exposición.
    '''
)

# 3) JSON: además del texto, ya trae metadata estructurada.
structured_policy = {
    "document_id": "POL-BEN-007",
    "title": "Beneficio de capacitación",
    "text": (
        "La empresa reintegra cursos aprobados previamente por el líder. "
        "La solicitud debe incluir programa, costo y relación con el rol. "
        "El reintegro se presenta dentro de los 30 días posteriores a la finalización."
    ),
    "metadata": {
        "topic": "capacitación",
        "department": "RRHH",
        "active": True,
        "version": "2026.1",
    },
}
write_file(
    CORPUS_DIR / "beneficio_capacitacion.json",
    json.dumps(structured_policy, ensure_ascii=False, indent=2),
)

# 4) PDF: ReportLab se usa SOLO para crear el archivo de prueba.
# Más adelante PyPDF será quien lo lea y extraiga su texto.
pdf_created = False
if HAS_REPORTLAB:
    from reportlab.pdfgen import canvas
    pdf_path = CORPUS_DIR / "politica_reuniones.pdf"
    document = canvas.Canvas(str(pdf_path))
    document.drawString(72, 760, "Politica de reuniones")
    document.drawString(
        72, 735,
        "Las reuniones de mas de 30 minutos deben incluir agenda y objetivo."
    )
    document.drawString(
        72, 710,
        "La persona convocante registra decisiones y responsables al finalizar."
    )
    # save() materializa el archivo PDF en disco.
    document.save()
    pdf_created = True
else:
    print("⚠ ReportLab no está disponible: se continúa sin crear el PDF de ejemplo.")

print({
    "markdown": True,
    "text": True,
    "json": True,
    "pdf_created": pdf_created,
})

✓ data/retrieval_corpus/faq_rrhh.md
✓ data/retrieval_corpus/respuesta_incidentes.txt
✓ data/retrieval_corpus/beneficio_capacitacion.json
{'markdown': True, 'text': True, 'json': True, 'pdf_created': True}


In [6]:
# Asocia metadata conocida a cada fuente para conservar contexto y habilitar filtros.
# Esta metadata viajará con cada chunk hasta los resultados de búsqueda.
# En producción podría venir de una DB, un manifest o un gestor documental;
# acá la declaramos explícitamente para que el laboratorio sea reproducible.
DOCUMENT_METADATA = {
    "faq_rrhh.md": {
        "document_id": "FAQ-RRHH-005",
        "topic": "vacaciones",
        "department": "RRHH",
        "active": True,
        "version": "2026.1",
    },
    "respuesta_incidentes.txt": {
        "document_id": "INC-SEC-006",
        "topic": "seguridad",
        "department": "Tecnología",
        "active": True,
        "version": "2026.1",
    },
    "politica_reuniones.pdf": {
        "document_id": "POL-MTG-008",
        "topic": "reuniones",
        "department": "Operaciones",
        "active": True,
        "version": "2026.1",
    },
}

# Carga Markdown o TXT y los convierte al contrato documental común.
def load_markdown_or_text(path: Path) -> list[dict]:
    meta = DOCUMENT_METADATA[path.name]
    return [{
        "document_id": meta["document_id"],
        "source": path.name,
        "page": 1,
        "text": path.read_text(encoding="utf-8"),
        "metadata": {
            key: value for key, value in meta.items()
            if key != "document_id"
        },
    }]

# El JSON ya trae text + metadata; lo adaptamos al mismo contrato común.
def load_json_document(path: Path) -> list[dict]:
    data = json.loads(path.read_text(encoding="utf-8"))
    return [{
        "document_id": data["document_id"],
        "source": path.name,
        "page": 1,
        "text": data["text"],
        "metadata": data["metadata"],
    }]

# Extrae el texto de un PDF página por página usando PyPDF.
# Cada página se transforma en una unidad documental antes del chunking.
def load_pdf(path: Path) -> list[dict]:
    if not HAS_PYPDF:
        print(f"⚠ Se omite {path.name}: falta pypdf")
        return []
    from pypdf import PdfReader
    meta = DOCUMENT_METADATA[path.name]
    documents = []
    for page_number, page in enumerate(PdfReader(str(path)).pages, start=1):
        text = (page.extract_text() or "").strip()
        if text:
            documents.append({
                "document_id": meta["document_id"],
                "source": path.name,
                "page": page_number,
                "text": text,
                "metadata": {
                    key: value for key, value in meta.items()
                    if key != "document_id"
                },
            })
    return documents

# Dispatcher: elige el loader por extensión y devuelve siempre el mismo contrato.
def load_documents(directory: Path) -> list[dict]:
    documents = []
    for path in sorted(directory.iterdir()):
        suffix = path.suffix.lower()
        if suffix in {".md", ".txt"}:
            documents.extend(load_markdown_or_text(path))
        elif suffix == ".json":
            documents.extend(load_json_document(path))
        elif suffix == ".pdf":
            documents.extend(load_pdf(path))
        else:
            print(f"⚠ Formato no soportado, se omite: {path.name}")
    return documents

loaded_documents = load_documents(CORPUS_DIR)

show_table([
    {
        "document_id": doc["document_id"],
        "source": doc["source"],
        "page": doc["page"],
        "words": len(doc["text"].split()),
        **doc["metadata"],
    }
    for doc in loaded_documents
])

,document_id,source,page,words,topic,department,active,version
0,POL-BEN-007,beneficio_capacitacion.json,1,33,capacitación,RRHH,True,2026.1
1,FAQ-RRHH-005,faq_rrhh.md,1,45,vacaciones,RRHH,True,2026.1
2,POL-MTG-008,politica_reuniones.pdf,1,24,reuniones,Operaciones,True,2026.1
3,INC-SEC-006,respuesta_incidentes.txt,1,25,seguridad,Tecnología,True,2026.1


### Validaciones de ingesta

Normalizar no alcanza: antes de generar chunks o embeddings conviene verificar que cada documento cumpla el contrato mínimo.

Un loader no debería dejar avanzar documentos vacíos, IDs ausentes, fuentes sin identificar, páginas faltantes o metadata inconsistente.

**Idea de diseño:** fallar temprano y con un mensaje claro es mejor que descubrir el problema varios pasos después, durante embeddings o retrieval.


In [7]:
# Valida el contrato mínimo de ingesta antes de generar chunks o embeddings.
# Devuelve errores descriptivos en lugar de dejar fallar el pipeline más adelante.
def validate_documents(documents: list[dict]) -> list[str]:
    errors = []
    for index, doc in enumerate(documents):
        # Estos campos forman el contrato común de todos los loaders.
        for field in ("document_id", "source", "page", "text", "metadata"):
            if not doc.get(field):
                errors.append(f"documento {index}: falta {field}")
        # Metadata debe ser dict porque luego se usa para filtros estructurados.
        if not isinstance(doc.get("metadata"), dict):
            errors.append(f"documento {index}: metadata no es dict")
    return errors

loader_errors = validate_documents(loaded_documents)
assert not loader_errors, loader_errors
print(f"✓ {len(loaded_documents)} documentos normalizados sin errores")

✓ 4 documentos normalizados sin errores


## 3. Recuperar el corpus base de la Clase 5

Se combinan documentos nuevos con las políticas utilizadas en la clase anterior.

`data/policies.json` funciona como corpus fuente cuando está disponible. El baseline embebido en esta celda solo permite reconstruir **los documentos de ejemplo**, no reemplaza los módulos ni el trabajo realizado en Clase 5.

In [8]:
# Recupera el corpus fuente utilizado previamente y suma los documentos nuevos.
# El baseline embebido solo reconstruye datos de ejemplo si falta policies.json.
base_policies_path = DATA_DIR / "policies.json"
if base_policies_path.exists():
    base_documents = json.loads(base_policies_path.read_text(encoding="utf-8"))
else:
    base_documents = [
        {
            "document_id": "POL-VAC-001",
            "source": "politica_vacaciones.md",
            "page": 1,
            "text": (
                "Las vacaciones deben registrarse con 15 días corridos de anticipación. "
                "El líder revisa cobertura y confirma o rechaza."
            ),
            "metadata": {
                "topic": "vacaciones",
                "department": "RRHH",
                "active": True,
                "version": "2026.1",
            },
        },
        {
            "document_id": "POL-EXP-002",
            "source": "politica_gastos.md",
            "page": 1,
            "text": (
                "Los reintegros de gastos se presentan con comprobante válido "
                "dentro de los 30 días corridos desde el gasto."
            ),
            "metadata": {
                "topic": "gastos",
                "department": "Finanzas",
                "active": True,
                "version": "2026.1",
            },
        },
        {
            "document_id": "POL-SEC-003",
            "source": "politica_seguridad.md",
            "page": 2,
            "text": (
                "Ante exposición de una contraseña se cambia la clave, "
                "se revocan sesiones y se abre un incidente de seguridad."
            ),
            "metadata": {
                "topic": "seguridad",
                "department": "Tecnología",
                "active": True,
                "version": "2026.1",
            },
        },
        {
            "document_id": "POL-REM-004",
            "source": "politica_trabajo_remoto.md",
            "page": 1,
            "text": (
                "El trabajo remoto requiere acuerdo previo con el líder, "
                "registro interno y disponibilidad durante el horario acordado."
            ),
            "metadata": {
                "topic": "trabajo remoto",
                "department": "RRHH",
                "active": True,
                "version": "2026.1",
            },
        },
    ]

all_documents = base_documents + loaded_documents
print({
    "base_documents": len(base_documents),
    "loaded_documents": len(loaded_documents),
    "total_documents": len(all_documents),
})

{'base_documents': 4, 'loaded_documents': 4, 'total_documents': 8}


## 4. Chunking y caché de embeddings

Antes de indexar, cada documento se divide en chunks y cada chunk se transforma en un embedding.

Calcular embeddings puede ser costoso, por lo que conviene **no recalcular un vector si ya tenemos exactamente ese texto con el mismo modelo**.

La caché usa una clave derivada de:

`modelo + texto`

Por lo tanto:

- mismo modelo + mismo texto → **cache hit**;
- cambia el texto → **cache miss**;
- cambia el modelo → **cache miss**.

Esto evita mezclar accidentalmente vectores generados por modelos distintos.


### 4.1 Observar hits, misses y persistencia antes del módulo

Primero construimos una caché mínima para hacer visible el mecanismo:

1. buscar una clave inexistente → **miss**;
2. guardar el vector;
3. volver a buscarlo → **hit**;
4. persistir la caché en JSON;
5. crear una nueva instancia y comprobar que el vector sigue disponible.

Ese último paso demuestra **persistencia entre instancias**, no solo memoria dentro del mismo objeto.


In [9]:
# Primero construimos una caché mínima para observar hits, misses y persistencia.
import hashlib
import json
from pathlib import Path

class JsonEmbeddingCachePreview:
    def __init__(self, path: str | Path):
        self.path = Path(path)

        # Si el archivo existe, recuperamos entradas de una ejecución anterior.
        self.data = (
            json.loads(self.path.read_text(encoding="utf-8"))
            if self.path.exists()
            else {}
        )
        self.hits = 0
        self.misses = 0

    # La clave combina modelo + texto: cambiar cualquiera invalida el vector cacheado.
    def key(self, model_name: str, text: str) -> str:
        payload = f"{model_name}\0{text}".encode("utf-8")
        return hashlib.sha256(payload).hexdigest()

    def get(self, model_name: str, text: str):
        value = self.data.get(self.key(model_name, text))

        if value is None:
            self.misses += 1
        else:
            self.hits += 1

        return value

    def set(self, model_name: str, text: str, vector: list[float]) -> None:
        self.data[self.key(model_name, text)] = vector

    # Persistimos en JSON para que otra instancia pueda reutilizar el vector.
    def flush(self) -> None:
        self.path.write_text(
            json.dumps(self.data),
            encoding="utf-8",
        )

preview_cache_path = DATA_DIR / "embedding_cache_preview.json"

# Empezamos desde cero para que la demostración sea reproducible.
preview_cache_path.unlink(missing_ok=True)
preview_cache = JsonEmbeddingCachePreview(preview_cache_path)

# 1) No existe todavía: miss.
assert preview_cache.get("model-a", "texto") is None

# 2) Guardamos el vector y lo persistimos.
preview_cache.set("model-a", "texto", [0.1, 0.2])
preview_cache.flush()

# 3) En la misma instancia ya obtenemos un hit.
assert preview_cache.get("model-a", "texto") == [0.1, 0.2]

# 4) Otro modelo produce otra clave: miss.
assert preview_cache.get("model-b", "texto") is None

# 5) Creamos OTRA instancia leyendo el JSON desde disco.
reloaded_cache = JsonEmbeddingCachePreview(preview_cache_path)
assert reloaded_cache.get("model-a", "texto") == [0.1, 0.2]

{
    "first_instance": {
        "hits": preview_cache.hits,
        "misses": preview_cache.misses,
        "entries": len(preview_cache.data),
    },
    "reloaded_instance": {
        "hits": reloaded_cache.hits,
        "misses": reloaded_cache.misses,
        "entries": len(reloaded_cache.data),
    },
    "persisted_file": preview_cache_path.exists(),
}


{'first_instance': {'hits': 1, 'misses': 2, 'entries': 1},
 'reloaded_instance': {'hits': 1, 'misses': 0, 'entries': 1},
 'persisted_file': True}

### 4.2 Persistir la caché reutilizable

Ahora llevamos la idea anterior a un módulo reusable.

La versión completa agrega:

- lookup por `modelo + texto`;
- persistencia a JSON;
- estadísticas de `hits`, `misses` y `hit_rate`;
- cálculo en batch **solo para los textos faltantes**;
- reconstrucción del resultado en el mismo orden recibido.

> En este laboratorio cacheamos embeddings de **documentos** porque se reutilizan en muchas búsquedas. Las queries suelen ser dinámicas y no se cachean.


In [10]:
%%writefile ai_agent_project/src/ai_agent_course/embedding_cache.py
# Persiste la caché reutilizable que usará el provider durante la indexación.
from __future__ import annotations

import hashlib
import json
from pathlib import Path
from typing import Sequence


# El archivo permite reutilizar vectores incluso entre ejecuciones de la notebook.
class JsonEmbeddingCache:
    def __init__(self, path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.data = (
            json.loads(self.path.read_text(encoding="utf-8"))
            if self.path.exists()
            else {}
        )
        self.hits = 0
        self.misses = 0

    def key(self, model_name, text):
        payload = f"{model_name}\0{text}".encode("utf-8")
        return hashlib.sha256(payload).hexdigest()

    def get(self, model_name, text):
        value = self.data.get(self.key(model_name, text))
        if value is None:
            self.misses += 1
        else:
            self.hits += 1
        return value

    def set(self, model_name, text, vector):
        self.data[self.key(model_name, text)] = vector

    def flush(self):
        self.path.write_text(
            json.dumps(self.data),
            encoding="utf-8",
        )

    def stats(self):
        total = self.hits + self.misses
        return {
            "hits": self.hits,
            "misses": self.misses,
            "hit_rate": round(self.hits / total, 3) if total else 0.0,
            "entries": len(self.data),
        }


# Envuelve un provider de embeddings y consulta la caché antes de calcular.
class CachedEmbeddingProvider:
    def __init__(self, base, cache):
        self.base = base
        self.cache = cache
        self.model_name = base.model_name
        self.dimension = base.dimension

    def embed_documents(self, texts: Sequence[str]):
        texts = list(texts)

        # Conserva el orden original de los textos.
        output = [None] * len(texts)

        # Registramos qué posiciones necesitan cálculo real.
        missing_positions = []
        missing_texts = []

        for index, text in enumerate(texts):
            cached = self.cache.get(self.model_name, text)
            if cached is None:
                missing_positions.append(index)
                missing_texts.append(text)
            else:
                output[index] = cached

        # Calculamos EN BATCH únicamente los embeddings que no estaban cacheados.
        if missing_texts:
            vectors = self.base.embed_documents(missing_texts)
            for position, text, vector in zip(
                missing_positions, missing_texts, vectors
            ):
                output[position] = vector
                self.cache.set(self.model_name, text, vector)
            self.cache.flush()

        return list(output)

    def embed_query(self, text):
        # Las queries suelen ser dinámicas y poco repetitivas.
        # Priorizamos cachear documentos, que se reutilizan entre búsquedas.
        return self.base.embed_query(text)

Overwriting ai_agent_project/src/ai_agent_course/embedding_cache.py


In [11]:
# Envuelve el provider de embeddings con una caché persistente.
from ai_agent_course.chunking import document_chunks
from ai_agent_course.embeddings import build_embedding_provider
from ai_agent_course.embedding_cache import (
    CachedEmbeddingProvider,
    JsonEmbeddingCache,
)

# Usa embeddings reales solo cuando se solicitaron y la dependencia está disponible.
# Si no, se mantiene el provider determinístico para ejecutar el laboratorio sin descargas.
request_real_embeddings = (
    os.getenv("USE_REAL_EMBEDDINGS", "0") == "1"
    and importlib.util.find_spec("sentence_transformers") is not None
)
base_provider = build_embedding_provider(request_real_embeddings)
cache = JsonEmbeddingCache(DATA_DIR / "embedding_cache.json")
embedding_provider = CachedEmbeddingProvider(base_provider, cache)

# Aplica el mismo chunking a todo el corpus antes de indexar.
# target_words controla el tamaño aproximado; overlap_words preserva contexto entre bordes.
all_chunks = document_chunks(
    all_documents,
    target_words=45,
    overlap_words=6,
)

# Inspecciona una muestra para verificar trazabilidad y tamaño de los chunks.
show_table([
    {
        "chunk_id": item["chunk_id"],
        "document_id": item["document_id"],
        "words": len(item["text"].split()),
        "source": item["source"],
        "topic": item["metadata"].get("topic"),
    }
    for item in all_chunks[:8]
])
print("Total de chunks:", len(all_chunks))

/Users/gasparcartasso/Desktop/AI Developer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9661.27it/s]
/Users/gasparcartasso/Desktop/AI Developer/ai_agent_project/src/ai_agent_course/embeddings.py:164: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self._model.get_sentence_embedding_dimension()


,chunk_id,document_id,words,source,topic
0,POL-VAC-001:recursive:000:f05eeb6e,POL-VAC-001,42,politica_vacaciones.md,vacaciones
1,POL-VAC-001:recursive:001:2bc935bb,POL-VAC-001,41,politica_vacaciones.md,vacaciones
2,POL-EXP-002:recursive:000:d54378e3,POL-EXP-002,38,politica_gastos.md,gastos
3,POL-EXP-002:recursive:001:3ac2515d,POL-EXP-002,26,politica_gastos.md,gastos
4,POL-SEC-003:recursive:000:8dd616f6,POL-SEC-003,32,politica_seguridad.md,seguridad
5,POL-SEC-003:recursive:001:b6ee93f8,POL-SEC-003,39,politica_seguridad.md,seguridad
6,POL-REM-004:recursive:000:5f104097,POL-REM-004,44,politica_trabajo_remoto.md,trabajo remoto
7,POL-REM-004:recursive:001:45047657,POL-REM-004,20,politica_trabajo_remoto.md,trabajo remoto


Total de chunks: 12


## 5. Batch processing e indexación

Una vez obtenidos los chunks, hay que generar embeddings e indexarlos.

En lugar de enviar uno por uno, usamos **batches**:

`chunks → batch → embeddings → vector store`

El batch reduce overhead y aprovecha operaciones vectorizadas. Sin embargo:

- un batch demasiado grande puede agotar memoria o superar límites del provider;
- un batch demasiado chico genera más llamadas y overhead.

`BATCH_SIZE` no cambia el significado semántico del retrieval: es principalmente una decisión de **rendimiento y capacidad**.


In [12]:
# Construye el backend de retrieval y mide la indexación por lotes.
from ai_agent_course import retrieval as retrieval_module

# Preferimos Qdrant cuando está disponible; el fallback conserva el mismo contrato.
# Así el resto del laboratorio puede usar search/index/delete sin conocer el backend concreto.
try:
    build_retriever = retrieval_module.build_retriever
    retriever = build_retriever(
        use_qdrant=HAS_QDRANT and os.getenv("USE_QDRANT", "1") == "1",
        path=DATA_DIR / "qdrant",
        embedding_provider=embedding_provider,
    )
except (TypeError, RuntimeError):
    retriever = retrieval_module.InMemoryRetriever(
        embedding_provider=embedding_provider,
    )

# Un batch pequeño permite observar el efecto sin ocultar el flujo didáctico.
# En producción se calibra según memoria, provider y volumen.
BATCH_SIZE = 3
batch_rows = []
started_index = time.perf_counter()

# Indexa el corpus en lotes y registra latencia y uso de caché por batch.
# Los contadores de cache son acumulados para hacer visible el comportamiento.
for batch_number, start in enumerate(
    range(0, len(all_chunks), BATCH_SIZE),
    start=1,
):
    batch = all_chunks[start:start + BATCH_SIZE]
    batch_started = time.perf_counter()
    retriever.index(batch, batch_size=BATCH_SIZE)
    batch_rows.append({
        "batch": batch_number,
        "chunks": len(batch),
        "latency_ms": round((time.perf_counter() - batch_started) * 1000, 2),
        "cache_hits_total": cache.hits,
        "cache_misses_total": cache.misses,
    })

index_latency_ms = round(
    (time.perf_counter() - started_index) * 1000,
    2,
)

show_table(batch_rows)
print("Indexación total (ms):", index_latency_ms)
print("Backend:", retriever.health())

,batch,chunks,latency_ms,cache_hits_total,cache_misses_total
0,1,3,559.39,0,3
1,2,3,95.87,0,6
2,3,3,95.88,0,9
3,4,3,86.08,0,12


Indexación total (ms): 837.35
Backend: {'backend': 'in-memory-compatible', 'points': 11, 'embedding_model': 'paraphrase-multilingual-MiniLM-L12-v2', 'vector_size': 384, 'persistent': False}


### Segunda pasada: comprobar la caché

Volvemos a pedir embeddings para **exactamente los mismos textos**, sin reindexar.

La expectativa es:

- primera pasada → varios `misses`;
- segunda pasada → esos mismos textos deberían convertirse en `hits`.

Esto permite comprobar que la optimización existe de verdad y no es solo una abstracción en el código.


In [13]:
# Solicita nuevamente los mismos embeddings para comprobar que la caché evita recálculos.
# Guardamos las estadísticas antes y después para hacer visible el cambio.
cache_before = cache.stats()
_ = embedding_provider.embed_documents([item["text"] for item in all_chunks])
cache_after = cache.stats()

show_table([
    {"momento": "antes", **cache_before},
    {"momento": "después", **cache_after},
])

,momento,hits,misses,hit_rate,entries
0,antes,0,12,0.0,26
1,después,12,12,0.5,26


## 6. Payload indexes y filtros

La búsqueda semántica responde:

> “¿qué chunks son más parecidos a esta consulta?”

Los filtros agregan otra condición:

> “¿qué chunks son más parecidos **dentro del universo permitido**?”

Ejemplo: podemos buscar sobre seguridad, pero restringiendo a documentos activos del departamento de Tecnología.

En Qdrant, los campos filtrados con frecuencia pueden tener **payload indexes** para hacer esos filtros más eficientes. En el backend in-memory la operación es un `no-op`, pero se mantiene el mismo contrato.


In [14]:
# Crea índices sobre metadata frecuente y luego combina filtro estructurado + búsqueda vectorial.
retriever.create_payload_indexes(
    ["document_id", "topic", "department", "active", "version"]
)

# El filtro restringe el universo válido de búsqueda.
# No reemplaza la similitud semántica: ambas condiciones trabajan juntas.
# top_k indica cuántos candidatos pedimos; subir k suele mejorar Recall pero puede agregar ruido.
filtered_results = retriever.search(
    "¿qué hago si se expone una contraseña?",
    top_k=5,
    filters={"department": "Tecnología", "active": True},
)

show_table([
    {
        "rank": rank,
        "document_id": result.document_id,
        "score": round(result.score, 3),
        "topic": result.metadata.get("topic"),
        "source": result.source,
    }
    for rank, result in enumerate(filtered_results, start=1)
])

,rank,document_id,score,topic,source
0,1,INC-SEC-006,0.630,seguridad,respuesta_incidentes.txt
1,2,POL-SEC-003,0.489,seguridad,politica_seguridad.md
2,3,POL-SEC-003,0.120,seguridad,politica_seguridad.md


## 7. Actualización y eliminación segura

Un vector store no “entiende” automáticamente que una nueva versión reemplaza a una anterior.

Si simplemente indexamos la versión nueva, podemos terminar con:

`versión vieja + versión nueva`

compitiendo en los resultados.

Por eso usamos el patrón:

**borrar documento anterior → regenerar chunks → reindexar → verificar**

`document_id` identifica al documento lógico y se mantiene estable entre versiones. Los `chunk_id` pueden cambiar porque el contenido y la partición pueden cambiar.


In [15]:
# Simula una nueva versión de un documento ya indexado.
updated_remote = {
    "document_id": "POL-REM-004",
    "source": "politica_trabajo_remoto.md",
    "page": 1,
    "text": (
        "El trabajo remoto requiere acuerdo previo con el líder, registro en la "
        "herramienta interna y disponibilidad durante el horario acordado. "
        "Los cambios permanentes requieren una nueva validación de RRHH."
    ),
    "metadata": {
        "topic": "trabajo remoto",
        "department": "RRHH",
        "active": True,
        "version": "2026.2",
    },
}

# Patrón seguro:
# 1) borrar todos los chunks de la versión anterior;
# 2) regenerar chunks desde el contenido nuevo;
# 3) reindexar únicamente la versión vigente.
count_before_update = retriever.count()
retriever.delete_document("POL-REM-004")
updated_chunks = document_chunks(
    [updated_remote],
    target_words=45,
    overlap_words=6,
)
retriever.index(updated_chunks, batch_size=3)

# 4) Verificamos que la consulta recupere la versión nueva y no contenido obsoleto.
updated_search = retriever.search(
    "¿qué cambios en trabajo remoto requieren nueva validación de RRHH?",
    top_k=3,
)

show_table([
    {
        "rank": rank,
        "document_id": result.document_id,
        "version": result.metadata.get("version"),
        "score": round(result.score, 3),
        "preview": result.text[:130],
    }
    for rank, result in enumerate(updated_search, start=1)
])
print({
    "count_before": count_before_update,
    "count_after": retriever.count(),
})

,rank,document_id,version,score,preview
0,1,POL-REM-004,2026.2,0.799,El trabajo remoto requiere acuerdo previo con ...
1,2,POL-VAC-001,2026.1,0.465,Las solicitudes de vacaciones deben registrars...
2,3,FAQ-RRHH-005,2026.1,0.417,# Preguntas frecuentes de RRHH Para pedir vaca...


{'count_before': 11, 'count_after': 10}


In [16]:
# Comprueba que una eliminación realmente reduzca los puntos almacenados en el índice.
temporary_document = {
    "document_id": "TMP-DELETE-999",
    "source": "temporal.txt",
    "page": 1,
    "text": "Documento temporal de QA que debe eliminarse del índice.",
    "metadata": {
        "topic": "temporal",
        "department": "QA",
        "active": False,
        "version": "test",
    },
}

# Insertamos un documento temporal únicamente para probar el ciclo de borrado.
retriever.index(document_chunks([temporary_document]), batch_size=2)
count_with_temp = retriever.count()
retriever.delete_document("TMP-DELETE-999")
count_without_temp = retriever.count()

assert count_without_temp < count_with_temp
print({
    "before_delete": count_with_temp,
    "after_delete": count_without_temp,
    "deleted_points": count_with_temp - count_without_temp,
})

{'before_delete': 11, 'after_delete': 10, 'deleted_points': 1}


## 8. Golden dataset

Para evaluar retrieval necesitamos preguntas cuyo resultado esperado conozcamos de antemano.

Cada caso define:

- `query`: consulta de prueba;
- `relevant_document_ids`: documentos que consideramos relevantes;
- `category`: categoría para facilitar el análisis;
- `expects_evidence`: si el corpus debería contener evidencia suficiente.

Los `relevant_document_ids` funcionan como nuestro **ground truth**.

Evaluamos por `document_id`, no por coincidencia exacta de texto, porque un mismo documento puede estar dividido en varios chunks.


In [17]:
# Define un golden dataset pequeño con consultas y documentos relevantes conocidos.
GOLDEN_CASES = [
    {
        "case_id": "VAC-01",
        "query": "¿con cuánta anticipación pido vacaciones?",
        "relevant_document_ids": ["POL-VAC-001", "FAQ-RRHH-005"],
        "category": "vacaciones",
        "expects_evidence": True,
    },
    {
        "case_id": "EXP-01",
        "query": "¿cuál es el plazo para presentar un reintegro?",
        "relevant_document_ids": ["POL-EXP-002"],
        "category": "gastos",
        "expects_evidence": True,
    },
    {
        "case_id": "SEC-01",
        "query": "expuse mi contraseña, ¿qué acciones debo tomar?",
        "relevant_document_ids": ["POL-SEC-003", "INC-SEC-006"],
        "category": "seguridad",
        "expects_evidence": True,
    },
    {
        "case_id": "REM-01",
        "query": "¿cómo se registra el trabajo remoto?",
        "relevant_document_ids": ["POL-REM-004"],
        "category": "trabajo remoto",
        "expects_evidence": True,
    },
    {
        "case_id": "CAP-01",
        "query": "¿qué necesito para pedir reintegro de una capacitación?",
        "relevant_document_ids": ["POL-BEN-007"],
        "category": "capacitación",
        "expects_evidence": True,
    },
    {
        "case_id": "NONE-01",
        "query": "¿la empresa ofrece estacionamiento gratuito?",
        "relevant_document_ids": [],
        "category": "sin evidencia",
        "expects_evidence": False,
    },
]
# Lo mostramos antes de evaluar para que el criterio esperado sea explícito.
show_table(
    GOLDEN_CASES,
    ["case_id", "category", "query", "relevant_document_ids", "expects_evidence"],
)

,case_id,category,query,relevant_document_ids,expects_evidence
0,VAC-01,vacaciones,¿con cuánta anticipación pido vacaciones?,"[POL-VAC-001, FAQ-RRHH-005]",True
1,EXP-01,gastos,¿cuál es el plazo para presentar un reintegro?,[POL-EXP-002],True
2,SEC-01,seguridad,"expuse mi contraseña, ¿qué acciones debo tomar?","[POL-SEC-003, INC-SEC-006]",True
3,REM-01,trabajo remoto,¿cómo se registra el trabajo remoto?,[POL-REM-004],True
4,CAP-01,capacitación,¿qué necesito para pedir reintegro de una capa...,[POL-BEN-007],True
5,NONE-01,sin evidencia,¿la empresa ofrece estacionamiento gratuito?,[],False


## 9. Métricas de retrieval

No alcanza con mirar “si la búsqueda parece buena”. Medimos el ranking contra el golden dataset.

- **Precision@k:** de los documentos recuperados dentro del top-k, qué proporción es relevante.
- **Recall@k:** de todos los documentos relevantes esperados, qué proporción apareció dentro del top-k.
- **Hit Rate@k:** indica si apareció **al menos un** documento relevante dentro del top-k.
- **MRR (Mean Reciprocal Rank):** premia que el **primer documento relevante** aparezca lo más arriba posible.
- **Latencia:** cuánto tarda cada consulta.

### ¿Qué significa `k`?

En estas métricas, `k` representa la cantidad de **documentos lógicos únicos** que observamos.

Si dos chunks pertenecen al mismo `document_id`, no deberían ocupar dos posiciones lógicas al evaluar a nivel documento.

Para casos sin evidencia, analizamos score y umbral por separado.


### 9.1 Calcular métricas sobre un ranking conocido

Antes de evaluar el corpus real, verificamos las fórmulas con un ranking pequeño cuyo resultado podemos razonar manualmente.

In [18]:
# Implementa métricas independientes del backend.
# Evaluamos a nivel DOCUMENTO para no premiar ni penalizar chunks duplicados.
def unique_document_ids(results, k):
    ids = []

    # Recorremos el ranking hasta reunir k documentos ÚNICOS.
    # No cortamos primero k chunks porque varios podrían pertenecer al mismo documento.
    for result in results:
        if result.document_id not in ids:
            ids.append(result.document_id)

        if len(ids) == k:
            break

    return ids

# Precision@k: de los documentos recuperados, qué proporción es relevante.
def precision_at_k(results, relevant_ids, k):
    retrieved = unique_document_ids(results, k)
    if not retrieved:
        return 0.0

    relevant = set(relevant_ids)
    return sum(doc_id in relevant for doc_id in retrieved) / len(retrieved)

# Recall@k: de todos los documentos relevantes esperados, cuántos recuperamos.
def recall_at_k(results, relevant_ids, k):
    relevant = set(relevant_ids)
    if not relevant:
        return 0.0

    retrieved = set(unique_document_ids(results, k))
    return len(retrieved & relevant) / len(relevant)

# Hit Rate@k: 1 si apareció al menos un relevante; 0 si no apareció ninguno.
def hit_rate_at_k(results, relevant_ids, k):
    relevant = set(relevant_ids)
    retrieved = set(unique_document_ids(results, k))
    return float(bool(retrieved & relevant))

# Reciprocal Rank = 1 / posición lógica del primer documento relevante.
def reciprocal_rank(results, relevant_ids):
    relevant = set(relevant_ids)
    seen = set()
    document_rank = 0

    for result in results:
        if result.document_id in seen:
            continue

        seen.add(result.document_id)
        document_rank += 1

        if result.document_id in relevant:
            return 1.0 / document_rank

    return 0.0


In [19]:
# Verifica las fórmulas con un ranking conocido antes de aplicarlas al corpus real.
from types import SimpleNamespace

# Agregamos un chunk duplicado para demostrar que no ocupa otra posición lógica.
# Ranking lógico: irrelevante -> vacaciones -> gastos.
known_ranking = [
    SimpleNamespace(document_id="irrelevante"),
    SimpleNamespace(document_id="irrelevante"),
    SimpleNamespace(document_id="vacaciones"),
    SimpleNamespace(document_id="gastos"),
]

expected_ids = ["vacaciones"]

metric_preview = {
    "unique_docs@2": unique_document_ids(known_ranking, 2),
    "precision@2": precision_at_k(known_ranking, expected_ids, 2),
    "recall@2": recall_at_k(known_ranking, expected_ids, 2),
    "hit_rate@1": hit_rate_at_k(known_ranking, expected_ids, 1),
    "mrr": reciprocal_rank(known_ranking, expected_ids),
}

# top-2 lógico = [irrelevante, vacaciones]
# precision@2 = 1/2 = 0.5
# recall@2 = 1/1 = 1.0
# hit@1 = 0 porque el primer documento no es relevante
# RR = 1/2 = 0.5
assert metric_preview["unique_docs@2"] == ["irrelevante", "vacaciones"]
assert metric_preview["precision@2"] == 0.5
assert metric_preview["recall@2"] == 1.0
assert metric_preview["hit_rate@1"] == 0.0
assert metric_preview["mrr"] == 0.5

metric_preview


{'unique_docs@2': ['irrelevante', 'vacaciones'],
 'precision@2': 0.5,
 'recall@2': 1.0,
 'hit_rate@1': 0.0,
 'mrr': 0.5}

### 9.2 Persistir métricas ya verificadas

La evaluación queda separada del retriever: cualquier backend que devuelva el contrato común puede medirse con las mismas funciones.

In [20]:
%%writefile ai_agent_project/src/ai_agent_course/retrieval_eval.py
# Métricas reutilizables e independientes del backend.
# Trabajamos a nivel documento para no contar varias veces chunks del mismo origen.

def unique_document_ids(results, k):
    ids = []

    for result in results:
        if result.document_id not in ids:
            ids.append(result.document_id)

        if len(ids) == k:
            break

    return ids


def precision_at_k(results, relevant_ids, k):
    retrieved = unique_document_ids(results, k)
    if not retrieved:
        return 0.0

    relevant = set(relevant_ids)
    return sum(doc_id in relevant for doc_id in retrieved) / len(retrieved)


def recall_at_k(results, relevant_ids, k):
    relevant = set(relevant_ids)
    if not relevant:
        return 0.0

    retrieved = set(unique_document_ids(results, k))
    return len(retrieved & relevant) / len(relevant)


def hit_rate_at_k(results, relevant_ids, k):
    relevant = set(relevant_ids)
    retrieved = set(unique_document_ids(results, k))
    return float(bool(retrieved & relevant))


def reciprocal_rank(results, relevant_ids):
    relevant = set(relevant_ids)
    seen = set()
    document_rank = 0

    for result in results:
        if result.document_id in seen:
            continue

        seen.add(result.document_id)
        document_rank += 1

        if result.document_id in relevant:
            return 1.0 / document_rank

    return 0.0


Overwriting ai_agent_project/src/ai_agent_course/retrieval_eval.py


In [21]:
# Ejecuta el golden dataset contra el retriever y calcula métricas caso por caso.
TOP_K = 5
per_case_rows = []
raw_results_by_case = {}
latency_rows = []

for case in GOLDEN_CASES:
    started = time.perf_counter()

    # TOP_K es la cantidad máxima de candidatos que pedimos al backend.
    results = retriever.search(
        case["query"],
        top_k=TOP_K,
    )

    latency_ms = round(
        (time.perf_counter() - started) * 1000,
        3,
    )

    # Conservamos el ranking crudo para recalcular métricas sin repetir búsquedas.
    raw_results_by_case[case["case_id"]] = results

    # La latencia se registra para TODOS los casos, incluso el caso sin evidencia.
    latency_rows.append({
        "case_id": case["case_id"],
        "expects_evidence": case["expects_evidence"],
        "latency_ms": latency_ms,
    })

    # Las métricas de relevancia se calculan cuando existe ground truth positivo.
    if case["expects_evidence"]:
        per_case_rows.append({
            "case_id": case["case_id"],
            "category": case["category"],
            "top_docs": unique_document_ids(results, TOP_K),
            "precision@3": round(
                precision_at_k(results, case["relevant_document_ids"], 3),
                3,
            ),
            "recall@3": round(
                recall_at_k(results, case["relevant_document_ids"], 3),
                3,
            ),
            "hit@3": hit_rate_at_k(
                results,
                case["relevant_document_ids"],
                3,
            ),
            "mrr": round(
                reciprocal_rank(results, case["relevant_document_ids"]),
                3,
            ),
            "top_score": round(results[0].score, 3) if results else None,
            "latency_ms": latency_ms,
        })

show_table(per_case_rows)


,case_id,category,top_docs,precision@3,recall@3,hit@3,mrr,top_score,latency_ms
0,VAC-01,vacaciones,"[FAQ-RRHH-005, POL-VAC-001, POL-MTG-008, POL-B...",0.667,1.0,1.0,1.0,0.528,20.752
1,EXP-01,gastos,"[POL-BEN-007, FAQ-RRHH-005, POL-VAC-001, POL-S...",0.000,0.0,0.0,0.0,0.664,24.506
2,SEC-01,seguridad,"[INC-SEC-006, POL-SEC-003, FAQ-RRHH-005, POL-V...",0.667,1.0,1.0,1.0,0.557,123.426
3,REM-01,trabajo remoto,"[POL-REM-004, POL-VAC-001, POL-BEN-007, FAQ-RR...",0.333,1.0,1.0,1.0,0.636,124.442
4,CAP-01,capacitación,"[POL-BEN-007, POL-REM-004, FAQ-RRHH-005, POL-S...",0.333,1.0,1.0,1.0,0.586,15.965


### Agregados por `k`

Mirar un único `k` puede ocultar problemas.

Si Recall mejora mucho entre `k=1` y `k=5`, significa que el documento correcto **existe en el ranking**, pero aparece demasiado abajo.

Compararemos Precision, Recall y Hit Rate para `k = 1, 3, 5`.

**MRR se reporta aparte**, porque en esta implementación mide la posición del primer relevante sobre el ranking lógico completo y no cambia simplemente por mostrar otro `k`.


In [22]:
# Compara Precision, Recall y Hit Rate para distintos valores de k.
aggregate_rows = []
evidence_cases = [
    case
    for case in GOLDEN_CASES
    if case["expects_evidence"]
]

# Usamos exactamente los mismos rankings; solo cambia cuántos documentos únicos observamos.
for k in (1, 3, 5):
    precision_values = []
    recall_values = []
    hit_values = []

    for case in evidence_cases:
        results = raw_results_by_case[case["case_id"]]
        relevant = case["relevant_document_ids"]

        precision_values.append(precision_at_k(results, relevant, k))
        recall_values.append(recall_at_k(results, relevant, k))
        hit_values.append(hit_rate_at_k(results, relevant, k))

    aggregate_rows.append({
        "k": k,
        "mean_precision": round(statistics.mean(precision_values), 3),
        "mean_recall": round(statistics.mean(recall_values), 3),
        "hit_rate": round(statistics.mean(hit_values), 3),
    })

show_table(aggregate_rows)

# MRR se calcula una sola vez sobre el ranking lógico completo de cada caso.
mean_mrr = round(
    statistics.mean([
        reciprocal_rank(
            raw_results_by_case[case["case_id"]],
            case["relevant_document_ids"],
        )
        for case in evidence_cases
    ]),
    3,
)

print("Mean Reciprocal Rank (MRR):", mean_mrr)


,k,mean_precision,mean_recall,hit_rate
0,1,0.80,0.6,0.8
1,3,0.40,0.8,0.8
2,5,0.26,0.8,0.8


Mean Reciprocal Rank (MRR): 0.8


## 10. Caso sin evidencia suficiente

Un retriever siempre puede devolver “lo más parecido”, incluso cuando **ningún resultado es realmente útil**.

Por eso la capa de aplicación necesita una política para decidir si usa la evidencia o se abstiene.

### Muy importante: `score` NO es probabilidad

Un score de similitud como `0.70` **no significa “70% de confianza”**.

No existe un threshold universal válido para todos los RAG. El valor depende de:

- modelo de embeddings;
- corpus;
- métrica de similitud;
- estrategia de chunking;
- distribución de consultas.

En este laboratorio usamos un umbral **ilustrativo** para mostrar el mecanismo. En un sistema real debe calibrarse empíricamente con un dataset de evaluación.


In [23]:
# Evalúa una consulta cuya respuesta no existe en el corpus.
no_evidence_case = next(
    case for case in GOLDEN_CASES if not case["expects_evidence"]
)
no_evidence_results = raw_results_by_case[no_evidence_case["case_id"]]

score_rows = [
    {
        "rank": rank,
        "document_id": result.document_id,
        "score": round(result.score, 3),
        "topic": result.metadata.get("topic"),
        "preview": result.text[:120],
    }
    for rank, result in enumerate(no_evidence_results, start=1)
]
show_table(score_rows)

# UMBRAL DIDÁCTICO: sirve para demostrar la política de abstención.
# No debe copiarse como threshold universal; en producción se calibra con datos.
MIN_RETRIEVAL_SCORE = (
    0.28 if "deterministic" in embedding_provider.model_name else 0.45
)
# Solo consideramos evidencia utilizable si supera el umbral definido.
accepted = [
    result for result in no_evidence_results
    if result.score >= MIN_RETRIEVAL_SCORE
]

print({
    "threshold": MIN_RETRIEVAL_SCORE,
    "accepted_results": len(accepted),
    "decision": "usar evidencia" if accepted else "abstenerse",
})

,rank,document_id,score,topic,preview
0,1,POL-VAC-001,0.212,vacaciones,Las solicitudes de vacaciones deben registrars...
1,2,POL-SEC-003,0.196,seguridad,Las credenciales personales no deben compartir...
2,3,FAQ-RRHH-005,0.184,vacaciones,# Preguntas frecuentes de RRHH Para pedir vaca...
3,4,POL-VAC-001,0.160,vacaciones,cobertura operativa y las fechas solicitadas. ...
4,5,POL-REM-004,0.160,trabajo remoto,El trabajo remoto requiere acuerdo previo con ...


{'threshold': 0.45, 'accepted_results': 0, 'decision': 'abstenerse'}


### Latencia

La calidad no es la única dimensión del retrieval: también importa cuánto tarda.

Con pocas consultas interesa observar la distribución, no solo el promedio. En un benchmark real conviene registrar p50, p95 y p99 con muchas más muestras.

Acá usamos media, mediana y máximo como diagnóstico inicial y contamos **todas** las queries del golden dataset, incluyendo el caso sin evidencia.


In [24]:
# Resume la latencia observada durante TODAS las consultas del golden dataset.
latencies = [
    row["latency_ms"]
    for row in latency_rows
]

latency_summary = {
    "queries": len(latencies),
    "mean_ms": round(statistics.mean(latencies), 3),
    "median_ms": round(statistics.median(latencies), 3),
    "max_ms": round(max(latencies), 3),
}

latency_summary


{'queries': 6, 'mean_ms': 53.602, 'median_ms': 22.629, 'max_ms': 124.442}

## 11. Diagnóstico guiado de errores

Una métrica baja no dice por sí sola qué cambiar. El orden recomendado es:

1. verificar loader y metadata;
2. leer los chunks recuperados;
3. revisar filtros;
4. comparar estrategia y tamaño de chunk;
5. cambiar embedding o agregar búsqueda híbrida;
6. volver a medir con el mismo golden dataset.

In [25]:
# Convierte las métricas en diagnósticos simples para orientar el siguiente ajuste.
diagnostic_rows = []
# Priorizamos primero ausencia de evidencia, luego ranking y finalmente ruido.
for row in per_case_rows:
    if row["hit@3"] == 0:
        diagnosis = "no recupera evidencia: revisar corpus, chunking y embedding"
    elif row["mrr"] < 1:
        diagnosis = "evidencia presente pero rankea abajo"
    elif row["precision@3"] < 0.5:
        diagnosis = "hay evidencia, pero también mucho ruido"
    else:
        diagnosis = "baseline correcto"
    diagnostic_rows.append({
        "case_id": row["case_id"],
        "category": row["category"],
        "diagnosis": diagnosis,
    })

show_table(diagnostic_rows)

,case_id,category,diagnosis
0,VAC-01,vacaciones,baseline correcto
1,EXP-01,gastos,"no recupera evidencia: revisar corpus, chunkin..."
2,SEC-01,seguridad,baseline correcto
3,REM-01,trabajo remoto,"hay evidencia, pero también mucho ruido"
4,CAP-01,capacitación,"hay evidencia, pero también mucho ruido"


## 12. Tests del retriever

Hasta acá hicimos inspección interactiva. Ahora fijamos algunos comportamientos como tests automatizados.

- una celda de notebook ayuda a **explorar y comprender**;
- un test ayuda a **detectar regresiones** cuando el código cambia.

Vamos a probar filtros, trazabilidad, métricas por documento, eliminación y actualización.


### 12.1 Persistir tests de retrieval

Los tests no verifican solamente que la búsqueda “devuelva algo”.

Fijan contratos concretos:

1. un filtro restringe resultados sin perder trazabilidad;
2. chunks repetidos del mismo documento cuentan como una sola posición lógica;
3. eliminar un documento retira sus chunks del índice;
4. actualizar un documento reemplaza la versión anterior.


In [26]:
%%writefile ai_agent_project/tests/test_class06.py
# Tests de contratos de retrieval que deben sobrevivir a futuros refactors.
from ai_agent_course.embeddings import DeterministicSemanticEmbeddings
from ai_agent_course.retrieval import InMemoryRetriever
from ai_agent_course.retrieval_eval import (
    hit_rate_at_k,
    precision_at_k,
    reciprocal_rank,
)


def build_test_retriever(chunks):
    # Usamos un provider determinístico para que los tests no dependan de APIs o descargas.
    return InMemoryRetriever(
        chunks,
        embedding_provider=DeterministicSemanticEmbeddings(),
    )


def test_filter_and_traceability():
    # Un filtro debe limitar resultados sin perder source ni document_id.
    chunks = [
        {
            "chunk_id": "A:0",
            "document_id": "A",
            "text": "Vacaciones con anticipación.",
            "source": "a.md",
            "page": 1,
            "metadata": {"department": "RRHH", "active": True, "version": "1"},
        },
        {
            "chunk_id": "B:0",
            "document_id": "B",
            "text": "Política de seguridad.",
            "source": "b.md",
            "page": 1,
            "metadata": {"department": "Tecnología", "active": True, "version": "1"},
        },
    ]

    retriever = build_test_retriever(chunks)

    results = retriever.search(
        "licencia anual",
        top_k=2,
        filters={"department": "RRHH"},
    )

    assert [result.document_id for result in results] == ["A"]
    assert results[0].source == "a.md"


def test_metrics_use_unique_documents():
    # Dos chunks del mismo documento deben ocupar una sola posición lógica.
    class Result:
        def __init__(self, document_id):
            self.document_id = document_id

    results = [
        Result("A"),
        Result("A"),
        Result("B"),
    ]

    # Ranking lógico: A primero, B segundo.
    assert hit_rate_at_k(results, ["B"], 2) == 1.0
    assert precision_at_k(results, ["B"], 2) == 0.5
    assert reciprocal_rank(results, ["B"]) == 0.5


def test_delete_document_removes_its_chunks():
    # Eliminar un document_id debe retirar todos sus chunks.
    chunks = [
        {
            "chunk_id": "A:0",
            "document_id": "A",
            "text": "Contenido que será eliminado.",
            "source": "a.md",
            "page": 1,
            "metadata": {"department": "QA", "active": True, "version": "1"},
        },
        {
            "chunk_id": "B:0",
            "document_id": "B",
            "text": "Contenido que debe permanecer.",
            "source": "b.md",
            "page": 1,
            "metadata": {"department": "QA", "active": True, "version": "1"},
        },
    ]

    retriever = build_test_retriever(chunks)
    count_before = retriever.count()

    retriever.delete_document("A")

    assert retriever.count() < count_before

    remaining = retriever.search(
        "contenido",
        top_k=10,
    )

    assert all(
        result.document_id != "A"
        for result in remaining
    )


def test_update_replaces_previous_version():
    # Actualizar = borrar la versión anterior y luego indexar la nueva.
    old_chunk = {
        "chunk_id": "A:0:old",
        "document_id": "A",
        "text": "La política anterior no requería validación.",
        "source": "a.md",
        "page": 1,
        "metadata": {"department": "RRHH", "active": True, "version": "1"},
    }

    new_chunk = {
        "chunk_id": "A:0:new",
        "document_id": "A",
        "text": "La nueva política requiere validación de RRHH.",
        "source": "a.md",
        "page": 1,
        "metadata": {"department": "RRHH", "active": True, "version": "2"},
    }

    retriever = build_test_retriever([old_chunk])

    retriever.delete_document("A")
    retriever.index([new_chunk], batch_size=1)

    results = retriever.search(
        "validación de RRHH",
        top_k=5,
    )

    assert any(
        result.document_id == "A"
        and result.metadata.get("version") == "2"
        for result in results
    )

    assert all(
        result.metadata.get("version") != "1"
        for result in results
        if result.document_id == "A"
    )


Overwriting ai_agent_project/tests/test_class06.py


## 13. Reporte y handoff a RAG

El retriever ya fue evaluado. Ahora consolidamos en **un único reporte** la configuración, las métricas, el golden dataset y los chunks necesarios para que la Clase 7 pueda continuar sin depender de archivos JSON adicionales.

### Reporte final de evidencia

La clase genera **un único `class06_report.json`** como evidencia consolidada y deja la continuidad preparada para la Clase 7.

- **Para el estudiante:** resume qué se construyó, con qué configuración y qué resultados obtuvo.
- **Para el docente:** permite revisar decisiones, métricas y validaciones sin recorrer todas las celdas.
- Los archivos que forman parte del funcionamiento real del proyecto (código, caché, tests, etc.) permanecen separados; el reporte evita duplicar JSON de evidencia.

**Contenido principal del reporte:** backend, embeddings, caché, métricas por caso, agregados por `k`, MRR, latencia, caso sin evidencia, golden dataset y chunks de handoff.


In [27]:
# Consolida la evidencia de la práctica y prepara el handoff hacia la Clase 7.
backend_health = retriever.health()

# Exporta el corpus lógico final: el reporte no necesita duplicar los vectores.
try:
    final_chunks = retriever.export_chunks()
except AttributeError:
    final_chunks = [
        item
        for item in all_chunks
        if item["document_id"] != "POL-REM-004"
    ] + updated_chunks

# Un único report reúne configuración, métricas y datos mínimos de continuidad.
report = {
    "class": 6,

    # Caso transversal que conecta esta clase con el proyecto incremental.
    "case_study": {
        "name": "agente de gestión asistida de emails",
        "stage": "retrieval",
        "summary": "Recuperar la política relevante para una consulta o email y medir si el retriever encuentra la evidencia correcta.",
    },
    "embedding": {
        "model": embedding_provider.model_name,
        "dimension": embedding_provider.dimension,
        "mode": "real" if request_real_embeddings else "smoke",
    },
    "retrieval": {
        "backend": backend_health,
        "documents": len(all_documents),
        "chunks": retriever.count(),
        "index_latency_ms": index_latency_ms,
        "cache": cache.stats(),
    },
    "evaluation": {
        "per_case": per_case_rows,
        "aggregate_by_k": aggregate_rows,
        "mrr": mean_mrr,
        "diagnostics": diagnostic_rows,
        "no_evidence": {
            "query": no_evidence_case["query"],
            "threshold": MIN_RETRIEVAL_SCORE,
            "accepted_results": len(accepted),
            "note": "Threshold didáctico: debe calibrarse para cada configuración real.",
        },
        "latency": latency_summary,
    },
    # La Clase 7 reutiliza estos datos para construir RAG sin generar JSON adicionales.
    "handoff": {
        # La Clase 7 recupera estos chunks desde el mismo reporte.
        "chunks": final_chunks,
        "golden_dataset": GOLDEN_CASES,
    },
    "continuity": {
        "increment": "ingesta, filtros y evaluación de retrieval",
        "next_class": 7,
    },
}

report_path = ARTIFACTS_DIR / "class06_report.json"

write_file(
    report_path,
    json.dumps(
        report,
        ensure_ascii=False,
        indent=2,
    ),
)

print("✓ Reporte Clase 6 generado:", report_path.relative_to(PROJECT_DIR))
{
    "class": 6,
    "report": str(report_path.relative_to(PROJECT_DIR)),
    "next_class": 7,
}

✓ artifacts/class06_report.json
✓ Reporte Clase 6 generado: artifacts/class06_report.json


{'class': 6, 'report': 'artifacts/class06_report.json', 'next_class': 7}

## Cierre

Un retriever está listo para conectarse a RAG cuando:

- ingiere formatos distintos bajo un contrato común;
- conserva `document_id`, fuente, página y metadata;
- recupera evidencia relevante de forma **medible**;
- filtra sin mezclar dominios;
- actualiza y elimina contenido correctamente;
- evita recalcular embeddings innecesariamente;
- detecta consultas sin evidencia suficiente;
- registra latencia y comportamiento;
- puede reproducirse desde artefactos y módulos versionados.

### La idea central de la Clase 6

Hasta acá todavía **no generamos una respuesta con un LLM**.

Construimos y evaluamos la parte **R — Retrieval** de RAG:

`query → embedding → búsqueda → ranking → evidencia`

En la Clase 7 conectaremos esa evidencia al contexto del modelo:

`Retrieval → Augmentation → Generation`


### Cierre opcional del retriever

La notebook no cierra Qdrant automáticamente para permitir reejecuciones. Cuando termines definitivamente, podés ejecutar `retriever.close()`.

## Continuidad del proyecto

El incremento queda guardado dentro de `ai_agent_project/` para que la siguiente clase lo importe y lo extienda.